<a href="https://colab.research.google.com/github/GENAURA/Accredian-Enterprize-clone1/blob/main/Assessment_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q rdflib sentence-transformers faiss-cpu

### 1. Build a Sample RDF Graph using rdflib
First, we will construct a simple RDF graph containing some factual triples.

In [2]:
from rdflib import Graph, URIRef, Literal, Namespace
from rdflib.namespace import RDF, RDFS

# Create a graph
g = Graph()

# Define namespaces
EX = Namespace("http://example.org/")
g.bind("ex", EX)

# Add some sample triples (subject, predicate, object)
triples = [
    (EX.Python, RDF.type, EX.ProgrammingLanguage),
    (EX.Python, RDFS.label, Literal("Python is a versatile programming language used for data science and web development.")),
    (EX.FAISS, RDF.type, EX.Library),
    (EX.FAISS, RDFS.label, Literal("FAISS is a library for efficient similarity search and clustering of dense vectors.")),
    (EX.RDFLib, RDF.type, EX.Library),
    (EX.RDFLib, RDFS.label, Literal("RDFLib is a Python library for working with RDF graphs.")),
    (EX.SentenceTransformers, RDF.type, EX.Library),
    (EX.SentenceTransformers, RDFS.label, Literal("Sentence-Transformers is a framework for state-of-the-art sentence, text and image embeddings."))
]

for s, p, o in triples:
    g.add((s, p, o))

print(f"Graph populated with {len(g)} triples.")

Graph populated with 8 triples.


### 2. Extract Text and Generate Embeddings
We will extract the literal descriptions (labels) from our graph and encode them into dense vectors using `sentence-transformers`.

In [3]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load a lightweight, high-performance model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Query the graph to extract entities and their descriptions
query_results = []
for s, p, o in g.triples((None, RDFS.label, None)):
    query_results.append({
        "entity": str(s),
        "description": str(o)
    })

# Extract descriptions and generate embeddings
descriptions = [item["description"] for item in query_results]
embeddings = model.encode(descriptions)

# Convert embeddings to float32 (required by FAISS)
embeddings_np = np.array(embeddings).astype('float32')
print(f"Generated embeddings shape: {embeddings_np.shape}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generated embeddings shape: (4, 384)


In [6]:
from rdflib import Graph, URIRef, Literal, Namespace
from rdflib.namespace import RDF, RDFS

# Create a new graph
g = Graph()

# Define namespaces
EX = Namespace("http://example.org/")
g.bind("ex", EX)

# Add sample triples (subject, predicate, object)
triples = [
    (EX.Python, RDF.type, EX.ProgrammingLanguage),
    (EX.Python, RDFS.label, Literal("Python is a versatile programming language used for data science and web development.")),
    (EX.FAISS, RDF.type, EX.Library),
    (EX.FAISS, RDFS.label, Literal("FAISS is a library for efficient similarity search and clustering of dense vectors.")),
    (EX.RDFLib, RDF.type, EX.Library),
    (EX.RDFLib, RDFS.label, Literal("RDFLib is a Python library for working with RDF graphs.")),
    (EX.SentenceTransformers, RDF.type, EX.Library),
    (EX.SentenceTransformers, RDFS.label, Literal("Sentence-Transformers is a framework for state-of-the-art sentence, text and image embeddings."))
]

for s, p, o in triples:
    g.add((s, p, o))

print(f"Graph populated successfully with {len(g)} triples.")

Graph populated successfully with 8 triples.


In [10]:
# Convert RDF triples into simple, natural sentences

# Helper function to clean URI strings for presentation
def clean_name(uri_or_node):
    # Extract fragment or last path component
    s = str(uri_or_node)
    if "#" in s:
        return s.split("#")[-1]
    elif "/" in s:
        return s.split("/")[-1]
    return s

print("Generated Simple Sentences from RDF:")
print("=" * 50)
for s, p, o in g:
    subj = clean_name(s)
    pred = clean_name(p)

    # Customize presentation based on standard RDF predicates
    if pred == "type":
        obj = clean_name(o)
        print(f"{subj} is a type of {obj}.")
    elif pred == "label":
        print(f"{subj} is described as: \"{o}\"")
    else:
        obj = clean_name(o)
        print(f"{subj} {pred} {obj}.")

Generated Simple Sentences from RDF:
SentenceTransformers is a type of Library.
FAISS is a type of Library.
Python is a type of ProgrammingLanguage.
FAISS is described as: "FAISS is a library for efficient similarity search and clustering of dense vectors."
RDFLib is described as: "RDFLib is a Python library for working with RDF graphs."
SentenceTransformers is described as: "Sentence-Transformers is a framework for state-of-the-art sentence, text and image embeddings."
RDFLib is a type of Library.
Python is described as: "Python is a versatile programming language used for data science and web development."


In [11]:
import faiss

# Get embedding dimension from our generated embeddings
dimension = embeddings_np.shape[1]

# Initialize the FAISS index for L2 (Euclidean) distance
index = faiss.IndexFlatL2(dimension)

# Add our document embeddings to the index
index.add(embeddings_np)

print(f"FAISS Index successfully created and populated with {index.ntotal} embeddings.")

FAISS Index successfully created and populated with 4 embeddings.


In [ ]:
# Prompt the user for a question, embed it, and query the FAISS index
user_query = input("Enter your search query: ")

# Generate embedding for the query
query_vector = model.encode([user_query]).astype('float32')

# Search FAISS index for top 1 match
k = 1
distances, indices = index.search(query_vector, k)

# Display the top matching RDF entity and description
print("\nSearch Results:")
print("=" * 50)
for idx, dist in zip(indices[0], distances[0]):
    if idx < len(query_results):
        match = query_results[idx]
        print(f"Matched Entity: {match['entity']}")
        print(f"Description:    {match['description']}")
        print(f"Distance:       {dist:.4f}")